# 🛡️ Geographic Threat Clustering (ST_CLUSTERDBSCAN)
**Groups threat origins within a 20-mile radius into clusters using BigQuery's `ST_CLUSTERDBSCAN` function.**

Three visualizations:
1. **Cluster bubble map** — where are clusters located? Size = threat count, color = rule diversity
2. **Stacked rule-breakdown bar** — which rules fire within each cluster?
3. **Attack pattern quadrant scatter** — IP breadth vs. threat intensity reveals the attack type

---

## 0 · Setup & BigQuery Connection

In [ ]:
# Install required packages (run once)
# !pip install google-cloud-bigquery db-dtypes plotly pandas

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_ID = "<your-gcp-project>"   # ← replace with your GCP project
DATASET    = "<your-dataset>"        # ← replace with your BigQuery dataset

TABLE_GEO  = f"`{PROJECT_ID}.{DATASET}.threat_transactions_geo`"

# Rule colour palette
RULE_COLORS = {
    "SUBNET_RATE":    "#c62828",
    "VELOCITY_BURST": "#e65100",
    "HIGH_SPEND":     "#f9a825",
}

# Rule diversity colour palette
RULE_DIV_COLORS = {
    "Single-rule (contained)":  "#34a853",
    "Two-rule (escalating)":    "#fbbc04",
    "All rules (coordinated)":  "#c62828",
    "Noise":                    "#9e9e9e",
}

# ── BigQuery client ───────────────────────────────────────────────────────────
# ADC (Application Default Credentials) is used automatically inside GCP.
# Outside GCP: run `gcloud auth application-default login` first.
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

print(f"Connected to project: {PROJECT_ID}")

---
## Query 7 · Geographic Threat Clustering — ST_CLUSTERDBSCAN (20-mile radius)
> Groups threat origins within 20 miles of each other into clusters. Each cluster gets a centroid,
> rule breakdown, IP list, and threat count — revealing **where** attacks are physically coordinated.
>
> `cluster_id = NULL` means an isolated point that didn't fit any cluster (DBSCAN noise).

In [ ]:
# ── Run the DBSCAN clustering query ───────────────────────────────────────────
sql_q7 = f"""
WITH geo_threats AS (
  SELECT *,
    ST_GEOGPOINT(longitude, latitude) AS geo
  FROM {TABLE_GEO}
  WHERE accepted = 0
    AND rule_name != 'VALIDATION'
    AND latitude IS NOT NULL
),
clustered AS (
  SELECT *,
    ST_CLUSTERDBSCAN(geo, 32186.88, 1) OVER () AS cluster_id
  FROM geo_threats
)
SELECT
  cluster_id,
  ANY_VALUE(country_name)               AS country,
  ANY_VALUE(city_name)                  AS city,
  ROUND(AVG(latitude),  4)              AS center_lat,
  ROUND(AVG(longitude), 4)              AS center_lon,
  COUNT(*)                              AS total_threats,
  COUNT(DISTINCT source_ip)             AS distinct_ips,
  COUNT(DISTINCT rule_name)             AS distinct_rules,
  ARRAY_AGG(DISTINCT rule_name)         AS rules_triggered,
  ARRAY_AGG(DISTINCT source_ip LIMIT 5) AS sample_ips
FROM clustered
GROUP BY cluster_id
ORDER BY total_threats DESC
"""
df_q7 = run_query(sql_q7)

# Tidy up array columns (BigQuery returns them as Python lists)
df_q7["rules_label"] = df_q7["rules_triggered"].apply(
    lambda r: " + ".join(sorted(r)) if isinstance(r, list) else str(r)
)
df_q7["sample_ips_label"] = df_q7["sample_ips"].apply(
    lambda ips: "\n".join(ips) if isinstance(ips, list) else str(ips)
)

# Separate noise (cluster_id IS NULL) from real clusters
df_clusters = df_q7[df_q7["cluster_id"].notna()].copy()
df_noise    = df_q7[df_q7["cluster_id"].isna()].copy()

df_clusters["label"] = df_clusters["cluster_id"].apply(lambda x: f"Cluster {int(x)}")
df_noise["label"]    = "Noise (isolated)"

div_labels = {1: "Single-rule (contained)", 2: "Two-rule (escalating)", 3: "All rules (coordinated)"}
df_clusters["rule_diversity"] = df_clusters["distinct_rules"].map(div_labels)
df_noise["rule_diversity"]    = "Noise"

print(f"Named clusters : {len(df_clusters)}")
print(f"Noise points   : {len(df_noise)}")
df_q7[["cluster_id", "country", "city", "total_threats",
       "distinct_ips", "distinct_rules", "rules_label"]].head(10)

Named clusters : 8
Noise points   : 0


,cluster_id,country,city,total_threats,distinct_ips,distinct_rules,rules_label
0,0,United Kingdom,London,268,66,3,['TXN_SUMMARY_30SEC' 'TXN_SUMMARY_1MIN' 'SUBNE...
1,1,United States,San Diego,220,65,1,['TXN_SUMMARY_30SEC']
2,2,China,Changchun,220,65,1,['TXN_SUMMARY_30SEC']
3,3,United States,Milton,13,1,1,['TXN_SUMMARY_30SEC']
4,6,Sweden,Linköping,12,2,2,['TXN_SUMMARY_30SEC' 'TXN_SUMMARY_1MIN']
5,4,Singapore,Singapore,8,1,1,['TXN_SUMMARY_1MIN']
6,5,United Kingdom,Boxford,8,1,1,['TXN_SUMMARY_1MIN']
7,7,Australia,Melbourne,8,1,1,['TXN_SUMMARY_1MIN']


In [ ]:
# ── Viz 1: Cluster bubble map ──────────────────────────────────────────────────
# Size = total_threats | Color = rule diversity (attack coordination level)
# Noise points shown as small grey markers

df_noise_map = df_noise.copy()
df_noise_map["total_threats"] = df_noise_map["total_threats"].clip(upper=2)  # keep noise small
df_map = pd.concat([df_clusters, df_noise_map], ignore_index=True)

fig_q7_map = px.scatter_geo(
    df_map,
    lat="center_lat",
    lon="center_lon",
    size="total_threats",
    color="rule_diversity",
    color_discrete_map=RULE_DIV_COLORS,
    hover_name="label",
    hover_data={
        "country": True, "city": True,
        "total_threats": True, "distinct_ips": True,
        "rules_label": True, "sample_ips_label": True,
        "center_lat": False, "center_lon": False, "rule_diversity": False,
    },
    size_max=55,
    projection="natural earth",
    title="Geographic Threat Clusters — DBSCAN 20-mile radius<br>"
          "<sup>Bubble size = threat count · Color = rule diversity (attack coordination level)</sup>",
    labels={
        "rule_diversity": "Attack Pattern", "total_threats": "Threat Count",
        "rules_label": "Rules Fired", "sample_ips_label": "Sample IPs",
        "distinct_ips": "Distinct IPs",
    },
    category_orders={"rule_diversity": [
        "All rules (coordinated)", "Two-rule (escalating)",
        "Single-rule (contained)", "Noise"
    ]},
)
fig_q7_map.update_layout(
    height=520,
    paper_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend=dict(
        title="Attack Pattern",
        yanchor="bottom", y=0.05, xanchor="left", x=0.01,
        bgcolor="rgba(255,255,255,0.85)", bordercolor="#dadce0", borderwidth=1,
    ),
    geo=dict(
        showframe=False, showcoastlines=True, coastlinecolor="#dadce0",
        bgcolor="#f8f9fa", landcolor="#e8f0fe", showland=True,
        showocean=True, oceancolor="#e3f2fd",
        showcountries=True, countrycolor="#c5cae9",
    ),
)
fig_q7_map.show()

In [ ]:
# ── Viz 2: Stacked bar — rule breakdown per cluster ────────────────────────────
# Second query to get per-rule counts within each cluster

sql_q7b = f"""
WITH geo_threats AS (
  SELECT *,
    ST_GEOGPOINT(longitude, latitude) AS geo
  FROM {TABLE_GEO}
  WHERE accepted = 0
    AND rule_name != 'VALIDATION'
    AND latitude IS NOT NULL
),
clustered AS (
  SELECT *,
    ST_CLUSTERDBSCAN(geo, 32186.88, 1) OVER () AS cluster_id
  FROM geo_threats
)
SELECT
  cluster_id,
  rule_name,
  ANY_VALUE(city_name) AS city,
  COUNT(*) AS rule_threats
FROM clustered
WHERE cluster_id IS NOT NULL
GROUP BY cluster_id, rule_name
ORDER BY cluster_id, rule_threats DESC
"""
df_q7b = run_query(sql_q7b)

city_map = df_clusters.set_index("cluster_id")["city"].to_dict()
df_q7b["bar_label"] = df_q7b.apply(
    lambda r: f"C-{int(r['cluster_id'])} ({city_map.get(r['cluster_id'], '?')})",
    axis=1,
)

fig_q7_bars = px.bar(
    df_q7b,
    x="rule_threats",
    y="bar_label",
    color="rule_name",
    color_discrete_map=RULE_COLORS,
    orientation="h",
    title="Cluster Rule Breakdown — Which rules fire in each geographic cluster",
    labels={"rule_threats": "Threat Count", "bar_label": "Cluster", "rule_name": "Rule"},
    text="rule_threats",
)
fig_q7_bars.update_traces(textposition="inside", insidetextanchor="middle")
fig_q7_bars.update_layout(
    height=max(320, len(df_q7b["bar_label"].unique()) * 52),
    barmode="stack",
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend_title_text="Rule", xaxis_title="Threat Count",
    yaxis=dict(autorange="reversed"),
)
fig_q7_bars.update_xaxes(showgrid=True, gridcolor="#dadce0")
fig_q7_bars.show()

In [ ]:
# ── Viz 3: Attack pattern quadrant scatter ─────────────────────────────────────
# X = distinct_ips (breadth) | Y = total_threats (intensity) | Size = distinct_rules
#
# Quadrant key:
#   Top-left   = few IPs, high blocks   → Velocity burst / High-spend (one bad actor)
#   Top-right  = many IPs, high blocks  → Subnet flood (distributed attack)
#   Bottom-left  = few IPs, low blocks  → Isolated incident
#   Bottom-right = many IPs, low blocks → Slow recon / scanner

med_ips     = df_clusters["distinct_ips"].median()
med_threats = df_clusters["total_threats"].median()

fig_q7_scatter = px.scatter(
    df_clusters,
    x="distinct_ips",
    y="total_threats",
    size="distinct_rules",
    color="rule_diversity",
    color_discrete_map=RULE_DIV_COLORS,
    hover_name="label",
    hover_data={
        "country": True, "city": True,
        "total_threats": True, "distinct_ips": True,
        "rules_label": True,
        "distinct_rules": False, "rule_diversity": False,
    },
    size_max=32,
    title="Attack Pattern Analysis — IP Spread vs. Threat Intensity per Cluster<br>"
          "<sup>Bubble size = distinct rules fired · Quadrants reveal attack type</sup>",
    labels={
        "distinct_ips":  "Distinct Source IPs  (Attack Breadth)",
        "total_threats": "Total Threats Blocked  (Intensity)",
        "rule_diversity": "Attack Pattern",
        "rules_label":   "Rules Fired",
    },
    category_orders={"rule_diversity": [
        "All rules (coordinated)", "Two-rule (escalating)", "Single-rule (contained)"
    ]},
)
fig_q7_scatter.add_vline(
    x=med_ips, line_dash="dot", line_color="#9e9e9e",
    annotation_text="median IPs", annotation_position="top"
)
fig_q7_scatter.add_hline(
    y=med_threats, line_dash="dot", line_color="#9e9e9e",
    annotation_text="median threats", annotation_position="right"
)
quadrant_annotations = [
    dict(x=0.02, y=0.98, xref="paper", yref="paper", showarrow=False,
         text="<b>Velocity / High-Spend</b><br>Few IPs, high blocks",
         font=dict(size=10, color="#c62828"), align="left",
         bgcolor="rgba(252,228,236,0.7)", borderpad=4),
    dict(x=0.98, y=0.98, xref="paper", yref="paper", showarrow=False,
         text="<b>Subnet Flood</b><br>Many IPs, high blocks",
         font=dict(size=10, color="#c62828"), align="right",
         bgcolor="rgba(252,228,236,0.7)", borderpad=4),
    dict(x=0.02, y=0.02, xref="paper", yref="paper", showarrow=False,
         text="<b>Isolated Incident</b><br>Few IPs, low blocks",
         font=dict(size=10, color="#34a853"), align="left",
         bgcolor="rgba(230,244,234,0.7)", borderpad=4),
    dict(x=0.98, y=0.02, xref="paper", yref="paper", showarrow=False,
         text="<b>Slow Recon</b><br>Many IPs, low blocks",
         font=dict(size=10, color="#e65100"), align="right",
         bgcolor="rgba(255,243,224,0.7)", borderpad=4),
]
fig_q7_scatter.update_layout(
    annotations=fig_q7_scatter.layout.annotations + tuple(quadrant_annotations),
    height=500,
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend_title_text="Attack Pattern",
)
fig_q7_scatter.update_xaxes(showgrid=True, gridcolor="#dadce0")
fig_q7_scatter.update_yaxes(showgrid=True, gridcolor="#dadce0")
fig_q7_scatter.show()